In [ ]:
!pip install yfinance -q

import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

In [ ]:
portfolio = pd.DataFrame({
    "ticker": [
        "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "JPM", "JNJ", "XOM", "COST", "PG", "V", "KO",
        "EME", "FIX", "DECK", "DUOL", "CAVA", "ALAB",
        "RDW", "CLSK", "LMB", "DFIN", "HIMS", "PRM",
        "SPY", "QQQ", "AGG", "TLT", "IEF", "GLD"
    ],
    "category": [
        "Large Cap", "Large Cap", "Large Cap", "Large Cap", "Large Cap", "Large Cap", "Large Cap", "Large Cap", "Large Cap", "Large Cap", "Large Cap", "Large Cap",
        "Mid Cap", "Mid Cap", "Mid Cap", "Mid Cap", "Mid Cap", "Mid Cap",
        "Small Cap", "Small Cap", "Small Cap", "Small Cap", "Small Cap", "Small Cap",
        "ETF", "ETF", "Bond ETF", "Bond ETF", "Bond ETF", "Commodity"
    ],
    "sector": [
        "Technology", "Technology", "Technology", "Consumer Discretionary", "Communication Services", "Financials", "Healthcare", "Energy", "Consumer Staples", "Consumer Staples", "Financials", "Consumer Staples",
        "Industrials", "Industrials", "Consumer Discretionary", "Communication Services", "Consumer Discretionary", "Technology",
        "Industrials", "Technology", "Industrials", "Financials", "Healthcare", "Materials",
        "Broad Market", "Technology ETF", "Fixed Income", "Fixed Income", "Fixed Income", "Commodity"
    ],
    "weight": [
        0.070, 0.070, 0.060, 0.050, 0.050, 0.040, 0.035, 0.035, 0.030, 0.030, 0.030, 0.025,
        0.025, 0.025, 0.020, 0.020, 0.020, 0.020,
        0.015, 0.015, 0.015, 0.015, 0.015, 0.015,
        0.030, 0.060, 0.050, 0.040, 0.035, 0.040
    ]
})

print(f"Total Portfolio Weight: {portfolio['weight'].sum():.1%}")

assert np.isclose(portfolio["weight"].sum(), 1.0), "Portfolio weights must sum to 100%."

portfolio

## Download Historical Price Data

This section pulls adjusted daily closing prices for all portfolio holdings using Yahoo Finance. The data begins after all selected holdings had publicly traded price history available.

In [ ]:
# Download historical price data

start_date = "2024-03-20"
end_date = "2026-01-01"

tickers = portfolio["ticker"].tolist()

prices = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=False
)["Close"]

prices.head()

## Calculate Daily Returns

Daily returns are calculated from adjusted closing prices and are used to compute portfolio-level performance and risk metrics.

In [ ]:
# Calculate daily returns for each holding

daily_returns = prices.pct_change().dropna()

daily_returns.head()

In [ ]:
# Match portfolio weights to the order of return columns

weights = portfolio.set_index("ticker").loc[daily_returns.columns, "weight"]

# Calculate daily portfolio returns

portfolio_returns = daily_returns.dot(weights)

portfolio_returns.head()

In [ ]:
print(f"Number of trading days analyzed: {len(portfolio_returns)}")
print(f"Average daily portfolio return: {portfolio_returns.mean():.4%}")

## Portfolio Performance and Risk Metrics

This section calculates key portfolio analytics, including total return, annualized return, volatility, Sharpe ratio, and maximum drawdown. These metrics help evaluate both performance and risk.

In [ ]:
# Portfolio performance and risk metrics

trading_days = 252
risk_free_rate = 0.04

total_return = (1 + portfolio_returns).prod() - 1

annualized_return = (1 + total_return) ** (trading_days / len(portfolio_returns)) - 1

annualized_volatility = portfolio_returns.std() * np.sqrt(trading_days)

sharpe_ratio = (annualized_return - risk_free_rate) / annualized_volatility

cumulative_returns = (1 + portfolio_returns).cumprod()

running_max = cumulative_returns.cummax()

drawdown = (cumulative_returns / running_max) - 1

max_drawdown = drawdown.min()

summary_metrics = pd.DataFrame({
    "Metric": [
        "Total Return",
        "Annualized Return",
        "Annualized Volatility",
        "Sharpe Ratio",
        "Maximum Drawdown"
    ],
    "Value": [
        f"{total_return:.2%}",
        f"{annualized_return:.2%}",
        f"{annualized_volatility:.2%}",
        f"{sharpe_ratio:.2f}",
        f"{max_drawdown:.2%}"
    ]
})

summary_metrics

## Growth of $1 Invested

This chart shows how one dollar invested in the portfolio would have grown over the analysis period.

In [ ]:
# Plot growth of $1 invested

plt.figure(figsize=(10, 5))
plt.plot(cumulative_returns)
plt.title("Growth of $1 Invested")
plt.xlabel("Date")
plt.ylabel("Portfolio Value")
plt.grid(True)
plt.show()

## Portfolio Drawdown

This chart shows the portfolio's decline from previous peaks over time. Maximum drawdown is one of the most important risk metrics because it measures the worst peak-to-trough loss.

In [ ]:
# Plot portfolio drawdown

plt.figure(figsize=(10, 5))
plt.plot(drawdown)
plt.title("Portfolio Drawdown")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.grid(True)
plt.show()

## Portfolio Allocation Analysis

This section analyzes how the portfolio is allocated across sectors, market capitalization groups, and asset classes. Allocation analysis helps identify concentration risk and diversification across the portfolio.

In [ ]:
# Sector allocation

sector_allocation = (
    portfolio
    .groupby("sector")["weight"]
    .sum()
    .sort_values(ascending=False)
)

sector_allocation

In [ ]:
plt.figure(figsize=(10, 5))
sector_allocation.plot(kind="bar")
plt.title("Portfolio Sector Allocation")
plt.xlabel("Sector")
plt.ylabel("Portfolio Weight")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y")
plt.show()

In [ ]:
# Market cap allocation

category_allocation = (
    portfolio
    .groupby("category")["weight"]
    .sum()
    .sort_values(ascending=False)
)

category_allocation

In [ ]:
plt.figure(figsize=(8, 5))
category_allocation.plot(kind="bar")
plt.title("Portfolio Allocation by Category")
plt.xlabel("Category")
plt.ylabel("Portfolio Weight")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y")
plt.show()

## Top and Bottom Performing Holdings

This section calculates each holding's total return over the analysis period and ranks the best and worst performers. This helps identify which securities contributed positively or negatively to portfolio performance.

In [ ]:
# Calculate total return for each holding

holding_returns = (prices.iloc[-1] / prices.iloc[0]) - 1

holding_performance = pd.DataFrame({
    "Ticker": holding_returns.index,
    "Total Return": holding_returns.values
})

holding_performance = holding_performance.merge(
    portfolio[["ticker", "category", "sector", "weight"]],
    left_on="Ticker",
    right_on="ticker",
    how="left"
).drop(columns="ticker")

holding_performance["Weighted Contribution"] = (
    holding_performance["Total Return"] * holding_performance["weight"]
)

holding_performance = holding_performance.sort_values(
    "Total Return",
    ascending=False
)

holding_performance.head()

In [ ]:
top_10_performers = holding_performance.head(10)

top_10_performers.style.format({
    "Total Return": "{:.2%}",
    "weight": "{:.2%}",
    "Weighted Contribution": "{:.2%}"
})

In [ ]:
bottom_10_performers = holding_performance.tail(10)

bottom_10_performers.style.format({
    "Total Return": "{:.2%}",
    "weight": "{:.2%}",
    "Weighted Contribution": "{:.2%}"
})

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(top_10_performers["Ticker"], top_10_performers["Total Return"])
plt.title("Top 10 Performing Holdings")
plt.xlabel("Ticker")
plt.ylabel("Total Return")
plt.grid(axis="y")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(bottom_10_performers["Ticker"], bottom_10_performers["Total Return"])
plt.title("Bottom 10 Performing Holdings")
plt.xlabel("Ticker")
plt.ylabel("Total Return")
plt.grid(axis="y")
plt.show()

## Performance Contribution

This section estimates each holding's contribution to total portfolio performance by multiplying its total return by its portfolio weight. This helps identify which positions had the greatest impact on the overall portfolio, not just which had the highest standalone return.

In [ ]:
# Rank holdings by weighted contribution to portfolio return

contribution_analysis = holding_performance.sort_values(
    "Weighted Contribution",
    ascending=False
).copy()

contribution_analysis.style.format({
    "Total Return": "{:.2%}",
    "weight": "{:.2%}",
    "Weighted Contribution": "{:.2%}"
})

In [ ]:
top_contributors = contribution_analysis.head(10)

top_contributors[[
    "Ticker",
    "category",
    "sector",
    "weight",
    "Total Return",
    "Weighted Contribution"
]].style.format({
    "weight": "{:.2%}",
    "Total Return": "{:.2%}",
    "Weighted Contribution": "{:.2%}"
})

In [ ]:
bottom_contributors = contribution_analysis.tail(10)

bottom_contributors[[
    "Ticker",
    "category",
    "sector",
    "weight",
    "Total Return",
    "Weighted Contribution"
]].style.format({
    "weight": "{:.2%}",
    "Total Return": "{:.2%}",
    "Weighted Contribution": "{:.2%}"
})

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(top_contributors["Ticker"], top_contributors["Weighted Contribution"])
plt.title("Top 10 Contributors to Portfolio Return")
plt.xlabel("Ticker")
plt.ylabel("Weighted Contribution")
plt.grid(axis="y")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(bottom_contributors["Ticker"], bottom_contributors["Weighted Contribution"])
plt.title("Bottom 10 Contributors to Portfolio Return")
plt.xlabel("Ticker")
plt.ylabel("Weighted Contribution")
plt.grid(axis="y")
plt.show()

## Correlation Analysis

This section analyzes how portfolio holdings move relative to one another. A correlation matrix helps evaluate diversification by showing whether holdings tend to move together or independently.

In [ ]:
# Correlation matrix

correlation_matrix = daily_returns.corr()

correlation_matrix

In [ ]:
plt.figure(figsize=(14, 10))
plt.imshow(correlation_matrix, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(correlation_matrix.columns)), correlation_matrix.columns, rotation=90)
plt.yticks(range(len(correlation_matrix.columns)), correlation_matrix.columns)
plt.title("Portfolio Holdings Correlation Matrix")
plt.show()

## Rolling Volatility

This section calculates 30-day rolling volatility to show how portfolio risk changed over time.

In [ ]:
# 30-day rolling annualized volatility

rolling_volatility = portfolio_returns.rolling(window=30).std() * np.sqrt(252)

plt.figure(figsize=(10, 5))
plt.plot(rolling_volatility)
plt.title("30-Day Rolling Annualized Volatility")
plt.xlabel("Date")
plt.ylabel("Volatility")
plt.grid(True)
plt.show()

## Portfolio Summary Dashboard

This final summary combines the key portfolio metrics into a single view.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

def show_kpi_dashboard():
    top_contributor = top_contributors.iloc[0]["Ticker"]
    bottom_contributor = bottom_contributors.iloc[0]["Ticker"]

    dashboard_html = f"""
    <div style="font-family: Arial, sans-serif; padding: 20px; border-radius: 12px; background-color: #f7f9fc;">
        <h2 style="text-align:center; color:#1f2937;">Institutional Portfolio Risk & Performance Dashboard</h2>
        <p style="text-align:center; color:#4b5563;">
            Interactive summary of portfolio performance, risk, diversification, and contribution metrics
        </p>

        <div style="display:grid; grid-template-columns: repeat(3, 1fr); gap:15px; margin-top:25px;">
            <div style="background:white; padding:18px; border-radius:10px; box-shadow:0 2px 6px rgba(0,0,0,0.08);">
                <h4 style="color:#6b7280;">Total Return</h4>
                <h2 style="color:#111827;">{total_return:.2%}</h2>
            </div>

            <div style="background:white; padding:18px; border-radius:10px; box-shadow:0 2px 6px rgba(0,0,0,0.08);">
                <h4 style="color:#6b7280;">Annualized Return</h4>
                <h2 style="color:#111827;">{annualized_return:.2%}</h2>
            </div>

            <div style="background:white; padding:18px; border-radius:10px; box-shadow:0 2px 6px rgba(0,0,0,0.08);">
                <h4 style="color:#6b7280;">Annualized Volatility</h4>
                <h2 style="color:#111827;">{annualized_volatility:.2%}</h2>
            </div>

            <div style="background:white; padding:18px; border-radius:10px; box-shadow:0 2px 6px rgba(0,0,0,0.08);">
                <h4 style="color:#6b7280;">Sharpe Ratio</h4>
                <h2 style="color:#111827;">{sharpe_ratio:.2f}</h2>
            </div>

            <div style="background:white; padding:18px; border-radius:10px; box-shadow:0 2px 6px rgba(0,0,0,0.08);">
                <h4 style="color:#6b7280;">Maximum Drawdown</h4>
                <h2 style="color:#b91c1c;">{max_drawdown:.2%}</h2>
            </div>

            <div style="background:white; padding:18px; border-radius:10px; box-shadow:0 2px 6px rgba(0,0,0,0.08);">
                <h4 style="color:#6b7280;">Holdings</h4>
                <h2 style="color:#111827;">{len(portfolio)}</h2>
            </div>

            <div style="background:white; padding:18px; border-radius:10px; box-shadow:0 2px 6px rgba(0,0,0,0.08);">
                <h4 style="color:#6b7280;">Sectors</h4>
                <h2 style="color:#111827;">{portfolio["sector"].nunique()}</h2>
            </div>

            <div style="background:white; padding:18px; border-radius:10px; box-shadow:0 2px 6px rgba(0,0,0,0.08);">
                <h4 style="color:#6b7280;">Top Contributor</h4>
                <h2 style="color:#047857;">{top_contributor}</h2>
            </div>

            <div style="background:white; padding:18px; border-radius:10px; box-shadow:0 2px 6px rgba(0,0,0,0.08);">
                <h4 style="color:#6b7280;">Bottom Contributor</h4>
                <h2 style="color:#b91c1c;">{bottom_contributor}</h2>
            </div>
        </div>
    </div>
    """

    display(HTML(dashboard_html))


def show_top_contributors():
    display(
        top_contributors[["Ticker", "category", "sector", "weight", "Total Return", "Weighted Contribution"]]
        .style.format({
            "weight": "{:.2%}",
            "Total Return": "{:.2%}",
            "Weighted Contribution": "{:.2%}"
        })
    )


def show_bottom_contributors():
    display(
        bottom_contributors[["Ticker", "category", "sector", "weight", "Total Return", "Weighted Contribution"]]
        .style.format({
            "weight": "{:.2%}",
            "Total Return": "{:.2%}",
            "Weighted Contribution": "{:.2%}"
        })
    )


def show_sector_allocation():
    plt.figure(figsize=(10, 5))
    sector_allocation.plot(kind="bar")
    plt.title("Portfolio Sector Allocation")
    plt.xlabel("Sector")
    plt.ylabel("Portfolio Weight")
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y")
    plt.show()


def show_category_allocation():
    plt.figure(figsize=(8, 5))
    category_allocation.plot(kind="bar")
    plt.title("Portfolio Allocation by Category")
    plt.xlabel("Category")
    plt.ylabel("Portfolio Weight")
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y")
    plt.show()


def show_drawdown():
    plt.figure(figsize=(10, 5))
    plt.plot(drawdown)
    plt.title("Portfolio Drawdown")
    plt.xlabel("Date")
    plt.ylabel("Drawdown")
    plt.grid(True)
    plt.show()


def show_rolling_volatility():
    rolling_volatility = portfolio_returns.rolling(window=30).std() * np.sqrt(252)

    plt.figure(figsize=(10, 5))
    plt.plot(rolling_volatility)
    plt.title("30-Day Rolling Annualized Volatility")
    plt.xlabel("Date")
    plt.ylabel("Volatility")
    plt.grid(True)
    plt.show()


def show_growth():
    plt.figure(figsize=(10, 5))
    plt.plot(cumulative_returns)
    plt.title("Growth of $1 Invested")
    plt.xlabel("Date")
    plt.ylabel("Portfolio Value")
    plt.grid(True)
    plt.show()


dashboard_selector = widgets.Dropdown(
    options=[
        "KPI Dashboard",
        "Growth of $1",
        "Drawdown",
        "Rolling Volatility",
        "Sector Allocation",
        "Category Allocation",
        "Top Contributors",
        "Bottom Contributors"
    ],
    value="KPI Dashboard",
    description="View:",
    style={"description_width": "initial"}
)

output = widgets.Output()

def update_dashboard(change):
    with output:
        clear_output(wait=True)

        if change["new"] == "KPI Dashboard":
            show_kpi_dashboard()
        elif change["new"] == "Growth of $1":
            show_growth()
        elif change["new"] == "Drawdown":
            show_drawdown()
        elif change["new"] == "Rolling Volatility":
            show_rolling_volatility()
        elif change["new"] == "Sector Allocation":
            show_sector_allocation()
        elif change["new"] == "Category Allocation":
            show_category_allocation()
        elif change["new"] == "Top Contributors":
            show_top_contributors()
        elif change["new"] == "Bottom Contributors":
            show_bottom_contributors()

dashboard_selector.observe(update_dashboard, names="value")

display(dashboard_selector, output)

with output:
    show_kpi_dashboard()

## Interactive Portfolio Dashboard

This interactive dashboard allows users to explore portfolio performance, risk, allocation, and contribution analysis from one interface.